# The five datasets at a glance

Summary of Endoscapes2023, CIFAR-10H, MHIST, AMi-Br and FER+ with example images: the train / val / test splits,
the labels (one yes/no label per class or criterion), and the raters' votes `k` out of `n` per image and label.

**Data.** The notebook reads a sample pack, one folder per dataset in `results/dataset_examples/<dataset>/`
(set `PACK` in the first code cell to use another location). Each folder holds:
- `table.npz`: the full label table of the dataset: `ids`, `k` and `n` (images × labels), `split`, `cluster`
  (video / slide / image), `labels` and, where each rater is known, `votes` (images × raters × labels);
- `examples.csv` and `img/`: a few hundred thumbnails, picked at random per label and vote pattern
  (columns `row` = row in the table, `id`, `split`, `label`, `bin` = vote pattern, `file`; `label` empty = a random image).

Terms: **majority-positive** = more than half the raters said yes; **unanimous** = all raters gave the same answer;
**vote pattern** = `k/n`, the number of raters who said yes out of the panel.

Sections: 1. overview table · 2. how much raters disagree · 3. one section per dataset · 4. interactive browser.


In [ ]:
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PACK = REPO / "results" / "dataset_examples"   # sample pack: one folder per dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.titleweight": "bold", "font.size": 10})
SPLITS = ("train", "val", "test")
SPLIT_COLOR = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
SEQ = "#2a78d6"

NAMES = {"endoscapes": "Endoscapes2023", "cifar10h": "CIFAR-10H", "mhist": "MHIST", "amibr": "AMi-Br",
         "ferplus": "FER+"}
INFO = {
    "endoscapes": dict(domain="surgery (laparoscopic video frames)", kind="multilabel", raters="3 surgeons",
                       native="854×480 colour", split_note="our re-split by video, 121 / 40 / 40 videos"),
    "cifar10h": dict(domain="general (tiny web photos)", kind="one choice per rater (≤ 1 positive label per image)", raters="crowd (MTurk)",
                     native="32×32 colour", split_note="ours, stratified by original class, 6,000 / 2,000 / 2,000"),
    "mhist": dict(domain="pathology (colorectal polyps, H&E)", kind="binary", raters="7 GI pathologists",
                  native="224×224 colour", split_note="official train/test; val = random 20% of train"),
    "amibr": dict(domain="pathology (breast-cancer mitoses, H&E)", kind="binary", raters="3 pathologists",
                  native="128×128 colour", split_note="ours, whole slides 60/20/20 within each source"),
    "ferplus": dict(domain="general (faces)", kind="one choice per rater (≤ 1 positive label per image)", raters="crowd taggers",
                    native="48×48 grey (stored 112×112)", split_note="official FER2013 Training / PublicTest / PrivateTest"),
}
ORDER = list(NAMES)


class Pack:
    def __init__(self, ds):
        d = PACK / ds
        z = np.load(d / "table.npz")
        self.ds, self.dir = ds, d
        self.ids, self.k, self.n, self.split = z["ids"], z["k"], z["n"], z["split"]
        self.cluster, self.labels = z["cluster"], list(z["labels"])
        self.votes = z["votes"] if "votes" in z.files else None
        self.ex = pd.read_csv(d / "examples.csv")
        self.fixed_panel = np.unique(self.n).size == 1 and self.n.flat[0] <= 10

    def img(self, row):
        return Image.open(self.dir / self.ex.loc[self.ex.row == row, "file"].iloc[0])

    def caption(self, row, top=3):
        k, n = self.k[row], self.n[row]
        if len(self.labels) <= 3:
            return "  ".join(f"{l} {k[j]:g}/{n[j]:g}" for j, l in enumerate(self.labels))
        best = np.argsort(-k)[:top]
        return "\n".join(f"{self.labels[j]} {k[j]:g}/{n[j]:g}" for j in best if k[j] > 0)


missing = [ds for ds in ORDER if not (PACK / ds / "table.npz").exists()]
assert not missing, f"sample pack not found for {missing} in {PACK}; set PACK to the folder that holds it"
P = {ds: Pack(ds) for ds in ORDER}

## 1. Overview

In [ ]:
rows = []
for ds, p in P.items():
    unanimous = ((p.k == 0) | (p.k == p.n)).mean(0)
    rows.append({
        "dataset": NAMES[ds], "domain": INFO[ds]["domain"], "labels": len(p.labels), "label type": INFO[ds]["kind"],
        "raters": INFO[ds]["raters"],
        "raters / image (min–median–max)": f"{p.n.min():g}–{np.median(p.n[:, 0]):g}–{p.n.max():g}",
        "images": f"{len(p.ids):,}",
        **{s: f"{(p.split == s).sum():,}" for s in SPLITS},
        "unanimous (mean over labels)": f"{unanimous.mean():.0%}",
        "native size": INFO[ds]["native"], "splits": INFO[ds]["split_note"],
    })
pd.DataFrame(rows).set_index("dataset").style.set_properties(**{"text-align": "left"})

One random image from each dataset (all images are resized to 256×256 for the encoder; Endoscapes is first centre-cropped
to 480×480, shown as the dashed box). Frame colour = split: train blue, val orange, test green; the caption gives the votes.

In [ ]:
def show(ax, p, row, caption=True, fontsize=7):
    im = p.img(row)
    ax.imshow(im, cmap="gray" if im.mode == "L" else None)
    if p.ds == "endoscapes":                               # centre crop 480 of 854 px wide
        w, h = im.size
        cw = w * 480 / 854
        ax.add_patch(Rectangle(((w - cw) / 2, 0), cw, h - 1, fill=False, ls="--", lw=0.8, ec="white"))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(SPLIT_COLOR[p.split[row]]); s.set_linewidth(2)
    if caption:
        ax.set_xlabel(p.caption(row), fontsize=fontsize, color="#0b0b0b")


rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 5, figsize=(15, 3.4), gridspec_kw={"width_ratios": [1.78, 1, 1, 1, 1]})
for ax, (ds, p) in zip(axes, P.items()):
    rand = p.ex.loc[p.ex.label.isna(), "row"].values
    show(ax, p, rng.choice(rand))
    ax.set_title(NAMES[ds])
plt.tight_layout()

## 2. How much do the raters disagree?

Left: share of images where the panel is **not** unanimous, per label (higher = more disagreement).
Right: how big the majority is, over all (image, label) pairs of a dataset where at least one rater said yes:
`max(k, n − k) / n`, so 1 = unanimous and 0.5 = an even split. The curve gives the share of pairs whose majority is at
most that size (votes are whole numbers, hence the steps); a curve further up and left = more disagreement.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1.1, 1]})
ylab, vals, cols = [], [], []
for i, (ds, p) in enumerate(P.items()):
    dis = 1 - ((p.k == 0) | (p.k == p.n)).mean(0)
    for j, l in enumerate(p.labels):
        ylab.append(f"{NAMES[ds]} · {l}"); vals.append(dis[j]); cols.append(i)
y = np.arange(len(vals))[::-1]
a1.barh(y, vals, color=[["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"][c] for c in cols], height=0.75)
a1.set_yticks(y, ylab, fontsize=8); a1.set_xlabel("share of images with a non-unanimous panel")
a1.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); a1.grid(axis="x", color="#e4e3df", lw=0.6); a1.set_axisbelow(True)
for yy, v in zip(y, vals):
    a1.text(v + 0.005, yy, f"{v:.0%}", va="center", fontsize=7, color="#52514e")

for i, (ds, p) in enumerate(P.items()):
    m = p.k > 0
    agree = (np.maximum(p.k, p.n - p.k) / p.n)[m]
    xs = np.sort(agree)
    a2.step(np.r_[0.5, xs], np.r_[0, np.arange(1, xs.size + 1) / xs.size], where="post", lw=2,
            color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"][i], label=NAMES[ds])
a2.set_xlabel("size of the majority, max(k, n−k)/n  (labels with k > 0)"); a2.set_ylabel("share of pairs with a majority this small or smaller"); a2.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
a2.legend(frameon=False, loc="upper left"); a2.grid(axis="y", color="#e4e3df", lw=0.6)
plt.tight_layout()

## 3. One dataset at a time

Each section shows: (a) how the votes fall per label, (b) majority-positive and unanimous rates per split,
(c) where each rater is known (Endoscapes, AMi-Br), how often each rater says yes, and (d) example images arranged by
label (rows) and vote pattern (columns). The frame colour of each image is its split; the caption gives the votes.

In [ ]:
def vote_plot(p, ax):
    if p.fixed_panel:                                   # share of images at each k, one group of bars per label
        n = int(p.n.flat[0]); w = 0.8 / len(p.labels)
        shades = plt.cm.Blues(np.linspace(0.45, 0.9, len(p.labels)))
        for j, l in enumerate(p.labels):
            share = [(p.k[:, j] == k).mean() for k in range(n + 1)]
            x = np.arange(n + 1) + (j - (len(p.labels) - 1) / 2) * w
            ax.bar(x, share, w * 0.92, color=shades[j], label=l)
        ax.set_xticks(range(n + 1), [f"{k}/{n}" for k in range(n + 1)]); ax.set_xlabel("raters saying yes")
        if len(p.labels) > 1: ax.legend(frameon=False)
    else:                                               # share of votes, images with k > 0, one line per label
        edges = np.linspace(0, 1, 11)
        cm = plt.cm.tab10 if len(p.labels) == 10 else plt.cm.Dark2
        for j, l in enumerate(p.labels):
            s = (p.k[:, j] / p.n[:, j])[p.k[:, j] > 0]
            h, _ = np.histogram(s, edges)
            ax.plot((edges[:-1] + edges[1:]) / 2, h / len(p.ids), marker="o", ms=4, lw=1.5, color=cm(j), label=l)
        ax.set_xlabel("share of raters choosing the label (images with ≥ 1 vote)")
        ax.legend(frameon=False, fontsize=7, ncol=2)
    ax.set_ylabel("share of all images"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_title("(a) vote patterns"); ax.grid(axis="y", color="#e4e3df", lw=0.6); ax.set_axisbelow(True)


def split_plot(p, ax, what):
    w = 0.8 / 3; x = np.arange(len(p.labels))
    for i, s in enumerate(SPLITS):
        m = p.split == s
        v = (p.k[m] > p.n[m] / 2).mean(0) if what == "maj" else ((p.k[m] == 0) | (p.k[m] == p.n[m])).mean(0)
        ax.bar(x + (i - 1) * w, v, w * 0.92, color=SPLIT_COLOR[s], label=f"{s} ({m.sum():,})")
    ax.set_xticks(x, p.labels, rotation=45 if len(p.labels) > 3 else 0, ha="right" if len(p.labels) > 3 else "center")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.grid(axis="y", color="#e4e3df", lw=0.6); ax.set_axisbelow(True)
    ax.set_title("(b) majority-positive" if what == "maj" else "(b) unanimous")
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3); ax.legend(frameon=False, fontsize=7, ncol=3, loc="upper center")


def rater_plot(p, ax):
    v = np.nanmean(p.votes, 0)                          # [R, K] yes-rate per rater and label
    w = 0.8 / v.shape[0]; x = np.arange(len(p.labels))
    shades = plt.cm.Greys(np.linspace(0.4, 0.85, v.shape[0]))
    for r in range(v.shape[0]):
        ax.bar(x + (r - (v.shape[0] - 1) / 2) * w, v[r], w * 0.92, color=shades[r], label=f"rater {r + 1}")
    ax.set_xticks(x, p.labels); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_title("(c) yes-rate per rater (all splits)")
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3); ax.legend(frameon=False, fontsize=7, ncol=3, loc="upper center")
    ax.grid(axis="y", color="#e4e3df", lw=0.6); ax.set_axisbelow(True)


def stats(ds):
    p = P[ds]
    has_r = p.votes is not None
    fig, axes = plt.subplots(1, 4 if has_r else 3, figsize=(17 if has_r else 14, 3.6),
                             gridspec_kw={"width_ratios": [1.6, 1, 1] + ([1] if has_r else [])})
    vote_plot(p, axes[0]); split_plot(p, axes[1], "maj"); split_plot(p, axes[2], "unan")
    if has_r: rater_plot(p, axes[3])
    fig.suptitle(NAMES[ds], x=0.01, ha="left", fontweight="bold"); plt.tight_layout()


def grid(ds, per_cell=1, labels=None, seed=0, fontsize=7):
    p = P[ds]; rng = np.random.default_rng(seed)
    labels = labels or p.labels
    ex = p.ex.dropna(subset=["label"])
    bins = list(dict.fromkeys(ex.sort_values("bin", key=lambda b: b.str.extract(r"^(\d+)")[0].astype(int)).bin))
    aspect = 384 / 216 if ds == "endoscapes" else 1
    if len(labels) == 1:                                # one label: rows = vote patterns, per_cell images each
        l = labels[0]
        fig, axes = plt.subplots(len(bins), per_cell, squeeze=False, figsize=(1.6 * per_cell, 1.95 * len(bins)))
        for i, b in enumerate(bins):
            rows = ex[(ex.label == l) & (ex.bin == b)].row.values
            pick = rng.choice(rows, min(per_cell, len(rows)), replace=False)
            for q in range(per_cell):
                if q < len(pick): show(axes[i, q], p, pick[q], fontsize=fontsize)
                else: axes[i, q].axis("off")
            axes[i, 0].set_ylabel(b, fontsize=10, fontweight="bold", color="#0b0b0b")
        fig.suptitle(f"{NAMES[ds]} — rows: {l} votes · frame colour: split", x=0.01, ha="left", fontsize=10)
        plt.tight_layout(); return
    fig, axes = plt.subplots(len(labels), len(bins) * per_cell, squeeze=False,
                             figsize=(1.55 * aspect * len(bins) * per_cell, 1.95 * len(labels)))
    for i, l in enumerate(labels):
        for c, b in enumerate(bins):
            rows = ex[(ex.label == l) & (ex.bin == b)].row.values
            pick = rng.choice(rows, min(per_cell, len(rows)), replace=False) if len(rows) else []
            for q in range(per_cell):
                ax = axes[i, c * per_cell + q]
                if q < len(pick): show(ax, p, pick[q], fontsize=fontsize)
                else: ax.axis("off")
                if i == 0 and q == 0: ax.set_title(b, loc="left", fontsize=9)
        axes[i, 0].set_ylabel(l, fontsize=10, fontweight="bold", color="#0b0b0b")
    fig.suptitle(f"{NAMES[ds]} — rows: label · columns: vote pattern for that label · frame colour: split",
                 x=0.01, ha="left", fontsize=10)
    plt.tight_layout()

### 3.1 Endoscapes2023 — surgery · 3 labels · multilabel · 3 raters

Frames from 201 laparoscopic cholecystectomy (keyhole gallbladder removal) videos, one video per patient (IHU
Strasbourg); the dissection phase only, one frame every 5 s → 11,090 frames. Three surgeons each judged the three
Critical View of Safety (CVS) criteria:
**C1** two structures (cystic duct and artery) seen entering the gallbladder; **C2** hepatocystic triangle cleared of fat
and fibrous tissue; **C3** lower third of the gallbladder separated from the liver bed. Most frames are negative
for all three; C2 is almost never positive. Split by video (no patient in two subsets).

In [ ]:
stats('endoscapes')

In [ ]:
grid('endoscapes', per_cell=2)   # change seed=… for other examples

### 3.2 CIFAR-10H — general · 10 classes · one choice per rater (≤ 1 positive label per image) · 47–63 raters (median 51)

The 10,000 CIFAR-10 *test* images (32×32 web photos), each labelled by about 50 Amazon Mechanical Turk workers who
picked one of 10 classes. We train one yes/no head per class (`k` = votes for the class). Most images are near
unanimous; disagreement sits in look-alike pairs (cat/dog, automobile/truck, deer/horse, airplane/ship).
The vote patterns are binned by share (0–20%, …, 80–100%); images with no vote for a class are left out of its row.

In [ ]:
stats('cifar10h')

In [ ]:
grid('cifar10h')   # change seed=… for other examples

### 3.3 MHIST — pathology · 1 label · binary · 7 raters

224×224 patches of H&E-stained colorectal polyp slides (Dartmouth-Hitchcock). Seven gastrointestinal pathologists
called each patch **SSA** (sessile serrated adenoma, precancerous) or HP (hyperplastic polyp, benign). The most
disputed dataset: only about a third of the patches are unanimous. Official test split; our validation is a
random 20% of the official train (no slide IDs, so patches of one slide may sit in train and val).

In [ ]:
stats('mhist')

In [ ]:
grid('mhist', per_cell=6)   # change seed=… for other examples

### 3.4 AMi-Br — pathology · 1 label · binary · 3 raters

128×128 patches, each centred on one mitotic figure (a dividing cell) in breast-cancer H&E slides, from MIDOG 2021
and TUPAC16. Three pathologists called each figure **atypical** (abnormal division) or normal; 22% are atypical by
majority. Our split keeps whole slides together (60/20/20 within each source).

In [ ]:
stats('amibr')

In [ ]:
grid('amibr', per_cell=6)   # change seed=… for other examples

### 3.5 FER+ — general · 8 emotions · one choice per rater (≤ 1 positive label per image) · 4–10 raters (median 10)

FER2013 faces (48×48 greyscale, our copy upscaled to 112×112), relabelled by 10 crowd taggers each (Microsoft FER+).
We use the 8 emotions as yes/no heads; "unknown" and "not a face" votes do not count towards `n`, so `n` varies.
Disgust and contempt are very rare (~0.3%). About 1 face in 6 has no emotion with a majority. Official splits.
Vote patterns binned by share as for CIFAR-10H.

In [ ]:
stats('ferplus')

In [ ]:
grid('ferplus', per_cell=2)   # change seed=… for other examples

## 4. Image browser

`browse(dataset, label, votes, split="all", n=8, seed=0)` shows `n` images from the pack for one label and vote
pattern (e.g. `browse("mhist", "SSA", "3/7")`); change `seed` for other images. `bins(dataset, label)` lists the
vote patterns available. The pack holds up to 12 images per label and vote pattern.
The second cell below adds dropdowns and a **Shuffle** button on top of `browse` (needs `ipywidgets` ≥ 8 in
JupyterLab 4 / Notebook 7; VS Code also works). Without `ipywidgets` it says so and the rest still works.


In [ ]:
import re


def bins(ds, label):
    """Vote patterns in the pack for one dataset and label."""
    ex = P[ds].ex
    b = ex.loc[ex.label == label, "bin"].unique()
    return sorted(b, key=lambda s: int(re.match(r"\d+", s).group()))


def browse(ds, label, votes, split="all", n=8, seed=0):
    """Show n pack images of one dataset, label and vote pattern (optionally one split)."""
    p = P[ds]
    rows = p.ex[(p.ex.label == label) & (p.ex.bin == votes)].row.values
    if split != "all":
        rows = rows[p.split[rows] == split]
    rows = np.random.default_rng(seed).permutation(rows)[:n]
    if not len(rows):
        print(f"no images in the pack for {ds} · {label} · {votes} · {split}; vote patterns: {bins(ds, label)}")
        return None
    aspect = 384 / 216 if ds == "endoscapes" else 1
    ncol = min(len(rows), 6 if aspect > 1 else 8)
    nrow = int(np.ceil(len(rows) / ncol))
    fig, axes = plt.subplots(nrow, ncol, squeeze=False, figsize=(2.3 * aspect * ncol, 2.6 * nrow))
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, rows):
        ax.axis("on"); show(ax, p, r, fontsize=8)
        ax.set_title(f"{p.ids[r]} · {p.split[r]}", fontsize=7, color="#52514e")
    fig.suptitle(f"{NAMES[ds]} · {label} · votes {votes}", x=0.01, ha="left")
    plt.tight_layout()
    return fig


browse("mhist", "SSA", "3/7");


In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display
except ImportError:
    W = None
    print("ipywidgets not installed: use browse(...) above")

if W is not None:
    dd_ds = W.Dropdown(options=[(NAMES[d], d) for d in ORDER], description="dataset")
    dd_lab = W.Dropdown(description="label")
    dd_bin = W.Dropdown(description="votes")
    dd_split = W.Dropdown(options=["all", *SPLITS], description="split")
    sl_n = W.IntSlider(value=8, min=1, max=12, description="images")
    bt = W.Button(description="Shuffle", icon="random")
    out = W.Output()
    state = {"seed": 0}

    def _on_ds(*_):
        dd_lab.options = P[dd_ds.value].labels
        _on_lab()

    def _on_lab(*_):
        if dd_lab.value is not None:
            dd_bin.options = bins(dd_ds.value, dd_lab.value)

    def _draw(*_):
        if dd_lab.value is None or dd_bin.value is None:
            return
        with out:
            out.clear_output(wait=True)
            fig = browse(dd_ds.value, dd_lab.value, dd_bin.value, dd_split.value, sl_n.value, state["seed"])
            if fig is not None:
                display(fig); plt.close(fig)

    def _shuffle(_):
        state["seed"] += 1; _draw()

    dd_ds.observe(_on_ds, "value"); dd_lab.observe(_on_lab, "value")
    for w in (dd_lab, dd_bin, dd_split, sl_n):
        w.observe(_draw, "value")
    bt.on_click(_shuffle)
    _on_ds()
    display(W.VBox([W.HBox([dd_ds, dd_lab, dd_bin]), W.HBox([dd_split, sl_n, bt]), out]))
    _draw()
